# DataFest: seis modelos y cuatro combinaciones
216 evaluaciones temporales. Los resultados incluidos provienen de los scripts, no de celdas ejecutadas aquí.

## 1. Ubicación
En Colab, descomprime primero el ZIP en `/content`. Ajusta la ruta si es necesario. Localmente, deja el notebook junto a `ejecutar.py`.

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/content/DataFest_6_modelos') if Path('/content/DataFest_6_modelos').exists() else Path.cwd()
assert (ROOT / 'ejecutar.py').exists(), 'Ajusta ROOT a la carpeta del proyecto'
os.chdir(ROOT)
print(ROOT)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

## 2. Revisar datos y configuración

In [ ]:
import pandas as pd, json
from src.datos import cargar_datos
train, test, sample = cargar_datos(ROOT / 'data')
display(train.groupby('mes').objetivo.agg(['size','mean']))
print(json.dumps(json.loads(Path('config_modelos.json').read_text()), indent=2))

## 3. Ejecutar las 216 pruebas
Seis modelos × cuatro combinaciones × nueve meses. Esta celda tarda varios minutos. Para consultar resultados ya calculados, pasa a la sección 4.

In [ ]:
from datetime import datetime
SALIDA = ROOT / ('ejecucion_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
subprocess.run([sys.executable, 'ejecutar.py', '--mes', 'ambos', '--interaccion', 'ambos', '--salida', str(SALIDA), '--hilos', '4'], check=True)
subprocess.run([sys.executable, 'analizar_variantes.py', '--salida', str(SALIDA)], check=True)

## 4. Examinar la ejecución incluida
Cambia RESULTADOS por SALIDA para abrir la ejecución que acabas de hacer.

In [ ]:
RESULTADOS = ROOT / 'resultados_completa'
display(pd.read_csv(RESULTADOS / 'comparacion_cuatro_variantes.csv'))
display(pd.read_csv(RESULTADOS / 'resumen_modelos.csv'))
r = pd.read_csv(RESULTADOS / 'comparacion_modelos.csv')
display(r.pivot(index='mes_validacion', columns='candidato', values='gini'))

In [ ]:
from IPython.display import Image, display, Markdown
display(Image(filename=str(RESULTADOS / 'gini_por_mes.png')))
display(Image(filename=str(RESULTADOS / 'conversiones_por_mes.png')))
display(Markdown((RESULTADOS / 'ANALISIS_VARIANTES.md').read_text()))

## 5. Entrega
Son probabilidades, no etiquetas binarias ni un valor de Gini. Diciembre no tiene respuestas conocidas.

In [ ]:
submission = pd.read_csv(RESULTADOS / 'submission.csv')
display(submission.head())
print(submission.shape)

## Interpretación
El número de unos y su tasa se presentan por mes. No hay etiquetas de diciembre ni años anteriores para verificar un efecto navideño. No agregamos tasas reales del mes evaluado a los predictores. El Gini mide ordenamiento, no exactitud de probabilidades.